In [4]:
%env DATA_PATH=../../../data
import pandas as pd
import plotly.express as px
import json

DATA_PATH = '../../../data'

env: DATA_PATH=../../../data


# Old

In [14]:
hills = pd.read_csv(f"{DATA_PATH}/exports/hills_archive_all.csv")
hills = hills[hills['Time'].notna()]
hills['Date'] = pd.to_datetime(hills['Date'], format='%Y/%m/%d')

In [15]:
hill3 = hills[hills['Hill'] == 3].copy()
hill2 = hills[hills['Hill'] == 2].copy()

In [16]:
# get date_driver_buggy_rollnumber key for each hill3
hill3['key'] = hill3.apply(lambda row: f"{row['Date'].date()}_{row['Driver']}_{row['Buggy']}_{row['Roll Number']}", axis=1)
hill2['key'] = hill2.apply(lambda row: f"{row['Date'].date()}_{row['Driver']}_{row['Buggy']}_{row['Roll Number']}", axis=1)

In [17]:

freerolls = pd.read_csv(f"{DATA_PATH}/exports/freerolls_archive.csv")
freerolls = freerolls[freerolls['Time'] < 150]
freerolls = freerolls[freerolls['Max Energy'] < 140]
freerolls['Date'] = pd.to_datetime(freerolls['Date'], format='%Y/%m/%d')
freerolls['Roll Number'] = freerolls['Roll Number'].fillna(0).astype(int)
freerolls['key'] = freerolls.apply(lambda row: f"{row['Date'].date()}_{row['Driver']}_{row['Buggy']}_{row['Roll Number']}", axis=1)
freerolls['raceday'] = freerolls['Date'].dt.year
freerolls.loc[freerolls['Date'].dt.month >= 5, 'raceday'] += 1
freerolls['raceday'] = freerolls['raceday'].astype(str)
freerolls['start_minute'] = freerolls['Roll Start Time'].apply(lambda x: int(x.split(':')[0]) * 60 + int(x.split(':')[1]) if pd.notna(x) else None)

In [18]:
with open(f'{DATA_PATH}/archive/rolls.json', 'r') as f:
  roll_metas = json.load(f)
roll_metas = pd.DataFrame.from_records(roll_metas)
roll_metas['rd'] = roll_metas.year + (roll_metas.month > 4)

In [19]:
buggy_map = {'kp': 'Kingpin II', 'seraph': 'Seraph', 'zuke': 'Zuke', 'fuko': 'Zulu Machafuko', 'menes': 'Menes', 'inviscid': 'Inviscid'}
def meta_roll_key(row):
  roll_num = int(row['roll']) if not pd.isna(row['roll']) else 0
  buggy = buggy_map.get(row['buggy'], None)
  driver = row['driver'].capitalize() if not pd.isna(row['driver']) else 'unknown'
  driver = 'Mei Xi' if driver == 'Meixi' else driver
  return f"{row['year']}-{row['month']:02d}-{row['day']:02d}_{driver}_{buggy}_{roll_num}"
roll_metas['key'] = roll_metas.apply(meta_roll_key, axis=1)
roll_metas.key

0       2014-09-21_Sussy_Seraph_1
1       2014-09-21_Sussy_Seraph_2
2       2014-09-21_Sussy_Seraph_3
3          2014-09-21_Feyi_Zuke_1
4          2014-09-21_Feyi_Zuke_2
                  ...            
1362    2026-03-29_unknown_None_0
1363    2026-03-29_unknown_None_0
1364    2026-03-29_unknown_None_0
1365    2026-03-29_unknown_None_0
1366    2026-03-29_unknown_None_0
Name: key, Length: 1367, dtype: object

In [20]:
freerolls = freerolls.merge(roll_metas, on='key', how='left', suffixes=('', '_meta'))

In [21]:
freerolls

,Id,Buggy,Driver,Date,Roll Number,Roll Start Time,Time,Max Speed,Max Energy,To Chute Energy Loss,...,bags,heat,wheel_type,wheels,hill_1,hill_2,hill_3,hill_4,hill_5,rd
0,1,Zuke,Alani,2025-09-20,4,12:19,105.0,9.56,79.23,125.77,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,7,Zuke,Audrey,2025-09-21,4,11:56,88.3,11.71,81.73,112.15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,8,Zuke,Audrey,2025-09-21,5,12:12,78.0,12.09,86.34,106.04,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10,Inviscid,Mei Xi,2025-09-21,2,11:27,65.1,14.44,93.52,86.80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,11,Inviscid,Mei Xi,2025-09-21,3,11:43,63.7,15.09,101.15,86.87,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594,1412,Inviscid,Mei Xi,2026-03-21,3,12:09,60.0,15.24,110.44,87.67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
595,1413,Kingpin II,Daisy,2026-03-22,1,11:38,62.5,15.17,100.61,77.61,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
596,1414,Seraph,Cadence,2026-03-22,2,11:58,57.6,15.81,122.07,93.04,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
597,1415,Seraph,Cadence,2026-03-22,4,12:22,56.7,15.99,124.31,91.48,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
cmap = {
  'Zuke': 'turquoise',
  'Inviscid': 'black',
  'Seraph': 'hotpink',
  'Kingpin II': 'darkorange',
}

In [23]:
px.scatter(freerolls, x='Max Energy', y='Time', color='raceday', hover_data=['Id', 'key', 'Max Speed'], color_discrete_map=cmap)

In [24]:
px.scatter(freerolls, x='Max Speed', y='Time', color='raceday', hover_data=['Id', 'key', 'Max Energy'], color_discrete_map=cmap)

In [25]:
px.scatter(freerolls, x='Max Speed', y='Chute Energy Loss', color='raceday', hover_data=['Id', 'key', 'Time'], color_discrete_map=cmap)

In [26]:
px.scatter(freerolls, x='Max Speed', y='Chute Energy Loss', color='Buggy', hover_data=['Id', 'key', 'Time'], color_discrete_map=cmap)

In [27]:
px.scatter(freerolls, x='Max Energy', y='To Chute Energy Loss', color='Buggy', hover_data=['Id', 'key', 'Time'], color_discrete_map=cmap)

In [28]:
px.scatter(freerolls, x='Max Energy', y='To Chute Energy Loss', color='raceday', hover_data=['Id', 'key', 'Time'], color_discrete_map=cmap)

In [29]:
px.scatter(freerolls, x='Max Energy', y='Max Speed', color='Buggy', hover_data=['Driver', 'Buggy', 'Date', 'Roll Number', 'Roll Start Time'], color_discrete_map=cmap)

In [30]:
px.scatter(freerolls, x='Max Energy', y='Max Speed', color='raceday', hover_data=['Id', 'key', 'Time'], color_discrete_map=cmap)

In [31]:
px.scatter(freerolls, x='Max Speed', y='Freeroll Energy Loss', color='Buggy', hover_data=['Driver', 'Buggy', 'Date', 'Roll Number', 'Roll Start Time'], color_discrete_map=cmap)

In [32]:
px.scatter(freerolls, x='Max Energy', y='Freeroll Energy Loss', color='Buggy', hover_data=['Driver', 'Buggy', 'Date', 'Roll Number', 'Roll Start Time'], color_discrete_map=cmap)

In [33]:
px.scatter(freerolls, x='Max Speed', y='Chute Energy Loss', color='Buggy', hover_data=['Id', 'key', 'Time'], color_discrete_map=cmap)

In [34]:
px.scatter(freerolls, x='Time', y='Chute Energy Loss', color='Buggy', hover_data=['Driver', 'Buggy', 'Date', 'Roll Number', 'Roll Start Time'], color_discrete_map=cmap)

In [35]:
rd_2456 = freerolls[freerolls.Date >= '2023-05-01']
px.scatter(rd_2456, x='Max Energy', y='Time', color='Buggy', hover_data=['Driver', 'Buggy', 'Date', 'Roll Number', 'Roll Start Time'], color_discrete_map=cmap)

In [36]:
px.scatter(freerolls, x='Max Speed', y='Pickup Speed', color='Buggy', hover_data=['Id', 'key', 'Time'], color_discrete_map=cmap)

## Hill 3

In [37]:
freerolls['Hill 3 Time'] = freerolls['Id'].map(hill3.set_index('Id')['Time'])
freerolls['Hill 3 Energy'] = freerolls['Max Energy'] - freerolls['Freeroll Energy Loss']
freerolls['Hill 3 Speed'] = (2*(freerolls['Hill 3 Energy'] + 73.6)) ** 0.5
freerolls['Hill 3 Pusher'] = freerolls['Id'].map(hill3.set_index('Id')['Pusher'])
# set all hill 3 times above 30 to nan
freerolls.loc[freerolls['Hill 3 Time'] > 30, 'Hill 3 Time'] = pd.NA

In [38]:
px.scatter(freerolls, x='Hill 3 Energy', y='Hill 3 Time', color='Buggy', hover_data=['key', 'Hill 3 Pusher', 'Time',  'Max Energy', 'To Chute Energy Loss', 'Chute Energy Loss', 'Hill 3 Speed'], color_discrete_map=cmap)

In [39]:
px.scatter(freerolls, x='Hill 3 Speed', y='Hill 3 Time', color='Buggy', hover_data=['key', 'Hill 3 Pusher', 'Time',  'Max Energy', 'To Chute Energy Loss', 'Chute Energy Loss', 'Hill 3 Speed'], color_discrete_map=cmap)

In [40]:
px.scatter(freerolls, x='Max Energy', y='Hill 3 Energy', color='Buggy', hover_data=['Id', 'key', 'Time'], color_discrete_map=cmap)

## Hill 2

In [41]:
freerolls['Crosswalk Speed'] = (2 * (freerolls['Max Energy'] - 62.1)) ** 0.5
freerolls['Hill 2 Time'] = freerolls['Id'].map(hill2.set_index('Id')['Time'])
freerolls['Hill 2 Pusher'] = freerolls['Id'].map(hill2.set_index('Id')['Pusher'])
freerolls.loc[freerolls['Hill 2 Time'] > 18, 'Hill 2 Time'] = pd.NA

In [42]:
px.scatter(freerolls, x='Crosswalk Speed', y='Time', color='raceday', hover_data=['Id', 'key', 'Max Speed', 'Hill 2 Time', 'Hill 2 Pusher'], color_discrete_map=cmap)

In [43]:
px.scatter(freerolls[~freerolls['Id'].isin([1073, 1071, 605, 873, 367])], x='Crosswalk Speed', y='Time', color='Buggy', hover_data=['Id', 'key', 'Max Speed', 'Hill 2 Pusher'], color_discrete_map=cmap)

In [44]:
px.scatter(freerolls, x='Hill 2 Time', y='Time', color='Buggy', hover_data=['Id', 'key', 'Max Speed', 'Hill 2 Pusher'], color_discrete_map=cmap)

In [45]:
px.scatter(freerolls, x='Hill 2 Time', y='Crosswalk Speed', color='Buggy', hover_data=['Id', 'key', 'Max Speed', 'Time', 'Hill 2 Pusher'], color_discrete_map=cmap)

In [46]:
px.scatter(freerolls, x='Crosswalk Speed', y='To Chute Energy Loss', color='Buggy', hover_data=['Id', 'key', 'Time'], color_discrete_map=cmap)

In [47]:
px.scatter(freerolls, x='Crosswalk Speed', y='Time', color='heat', hover_data=['Id', 'key', 'Time'], color_discrete_map=cmap)

In [48]:
px.scatter(freerolls, x='Crosswalk Speed', y='Max Speed', color='Buggy', hover_data=['Id', 'key', 'Time'], color_discrete_map=cmap)

In [49]:
px.scatter(freerolls, x='Crosswalk Speed', y='Max Speed', color='heat', hover_data=['Id', 'key', 'Time'], color_discrete_map=cmap)

# New

In [5]:
import numpy as np
from sqlalchemy import select
from db.database import File, RollStat, engine

# every RollStat quantity as `<quantity>` / `<quantity>.sd`, one row per roll
raw_stats = pd.read_sql(
  select(RollStat.roll_id, RollStat.quantity, RollStat.value, RollStat.sd, RollStat.status,
         File.type.label('source')).join(File, File.id == RollStat.source_id), engine)
raw_stats['source'] = np.where(raw_stats['source'].str.contains('racebox'), 'racebox', 'pnp')
raw_stats.loc[raw_stats['status'] != 'ok', ['value', 'sd']] = np.nan
# racebox preferred for the few rolls carrying both sources
raw_stats = raw_stats[(raw_stats['source'] == 'racebox')
                | ~raw_stats.groupby('roll_id')['source'].transform(lambda s: (s == 'racebox').any())]

roll_stats = raw_stats.pivot(index='roll_id', columns='quantity', values=['value', 'sd'])
roll_stats.columns = [q if kind == 'value' else f'{q}.sd' for kind, q in roll_stats.columns]
roll_stats = roll_stats.sort_index(axis=1)
roll_stats.insert(0, 'source', raw_stats.groupby('roll_id')['source'].first())
roll_stats = roll_stats.reset_index()

In [53]:
from db.database import Buggy, Driver, Roll, RollDate

STAT_NAMES = {
  'time.crosswalk-hill_3': 'time', 'time.crosswalk-chute_start': 'to_chute_time',
  'time.chute_start-hill_3': 'chute_time', 'time.crosswalk-stop_sign': 'to_stop_sign_time',
  'time.stop_sign-hill_3': 'from_stop_sign_time', 'time.hill_1-finish_line': 'roll_time',
  'time.hill_1-hill_2': 'hill_1_time', 'time.hill_2-crosswalk': 'hill_2_time',
  'time.hill_3-hill_4': 'hill_3_time', 'time.hill_4-hill_5': 'hill_4_time',
  'time.hill_5-finish_line': 'hill_5_time',
  'max_speed': 'max_speed', 'max_energy': 'max_energy',
  'speed.crosswalk': 'crosswalk_speed', 'speed.chute_start': 'chute_speed',
  'energy.crosswalk': 'crosswalk_energy', 'energy.chute_start': 'chute_energy',
  'eloss.crosswalk-chute_start': 'to_chute_energy_loss',
  'eloss.chute_start-hill_3': 'chute_energy_loss',
  'eloss.crosswalk-hill_3': 'freeroll_energy_loss',
  'pickup.speed': 'pickup_speed', 'pickup.arc': 'pickup_arc',
  'path.crosswalk-chute_start': 'to_chute_path', 'path.chute_start-hill_3': 'chute_path',
  'path.crosswalk-hill_3': 'freeroll_path',
}

meta = pd.read_sql(
  select(Roll.id.label('roll_id'), Driver.name.label('driver'), Buggy.name.label('buggy'),
         RollDate.year, RollDate.month, RollDate.day, RollDate.type.label('roll_type'),
         Roll.roll_number, Roll.start_time.label('roll_start_time'), Roll.mech_notes)
  .join(Driver, Driver.id == Roll.driver_id).join(Buggy, Buggy.id == Roll.buggy_id)
  .join(RollDate, RollDate.id == Roll.roll_date_id), engine)
meta['roll_type'] = meta['roll_type'].map(lambda t: t.value)
meta['roll_number'] = meta['roll_number'].astype('Int64')
meta['date'] = pd.to_datetime(meta[['year', 'month', 'day']])
meta['raceday'] = (meta['year'] + (meta['month'] >= 5)).astype('str')  # season the roll belongs to
meta = meta.drop(columns=['year', 'month', 'day'])

rolls = roll_stats.rename(columns={**STAT_NAMES,
                                   **{f'{q}.sd': f'{n}_sd' for q, n in STAT_NAMES.items()}})
rolls = meta.merge(rolls, on='roll_id', how='right')
rolls = rolls[['roll_id', 'date', 'raceday', 'roll_type', 'roll_number', 'roll_start_time',
               'driver', 'buggy', 'source', 'mech_notes']
              + [c for n in STAT_NAMES.values() for c in (n, f'{n}_sd')]]
rolls

,roll_id,date,raceday,roll_type,roll_number,roll_start_time,driver,buggy,source,mech_notes,...,pickup_speed,pickup_speed_sd,pickup_arc,pickup_arc_sd,to_chute_path,to_chute_path_sd,chute_path,chute_path_sd,freeroll_path,freeroll_path_sd
0,1,2025-09-20,2026,weekend,4,2025-09-20 12:19:57.814000,Alani,Zuke,pnp,,...,3.377390,0.344774,-36.215756,4.790611,552.032236,0.199820,194.981689,0.264908,747.938507,0.217109
1,6,2025-09-21,2026,weekend,2,2025-09-21 11:28:56.310000,Audrey,Zuke,pnp,,...,3.615731,0.117053,-16.237976,1.705124,553.434367,0.303135,198.545185,0.336639,752.688726,0.324686
2,7,2025-09-21,2026,weekend,4,2025-09-21 11:56:42.282000,Audrey,Zuke,pnp,,...,3.095781,0.412505,-9.708700,3.142142,553.519190,0.747411,200.898401,0.314659,755.140488,0.373134
3,8,2025-09-21,2026,weekend,5,2025-09-21 12:12:32.412000,Audrey,Zuke,pnp,,...,5.259842,0.182518,-10.353782,2.906295,552.915753,0.532398,201.345991,0.207365,755.483964,0.267061
4,9,2025-09-21,2026,weekend,1,2025-09-21 11:10:42.091999,Mei Xi,Inviscid,pnp,,...,2.902068,1.149369,28.759577,6.911661,554.918537,0.969916,196.582161,0.297238,751.959642,0.272772
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1114,1412,2026-03-21,2026,weekend,3,2026-03-21 12:09:00.000000,Mei Xi,Inviscid,racebox,,...,4.073510,0.023407,-0.845966,1.619338,557.968155,0.007012,194.153326,0.005375,752.672044,0.006734
1115,1413,2026-03-22,2026,weekend,2,2026-03-22 11:38:00.000000,Daisy,Kingpin II,racebox,no virb attached,...,5.164995,0.162911,15.191132,1.896971,559.869473,0.005953,199.193725,0.005840,759.656598,0.005416
1116,1414,2026-03-22,2026,weekend,3,2026-03-22 11:58:00.000000,Cadence,Seraph,racebox,no virb attached,...,6.021610,0.157453,12.233139,2.752332,555.700875,0.005927,200.674259,0.006816,756.985331,0.005009
1117,1415,2026-03-22,2026,weekend,4,2026-03-22 12:22:00.000000,Cadence,Seraph,racebox,no virb attached,...,5.744586,0.036799,22.821869,1.650265,556.489251,0.006229,201.086717,0.005021,758.197184,0.007029


In [58]:
print([r.split('\n') for r in rolls.mech_notes])

[[''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], ['Pass test'], [''], [''], ['Bagged (pass test)'], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], [''], 

In [69]:
import re

WHEEL = re.compile(r'^(\d{2})-([A-Z])(\d{2})?(-RDR)?$')

def _triples(text):
  """(batch year, type, profile) of every wheel id or wheel_type label in `text`, or None if any
  token is unreadable -- a set we cannot fully read must not look uniform."""
  out = set()
  for tok in re.split(r'[,/]', text.upper()):
    tok = re.sub(r'-([A-Z])O(\d)', r'-\g<1>0\g<2>', re.sub(r'[()?]', '', tok).strip())  # O/0 typo
    m = WHEEL.match(tok)
    if not m:
      return None
    out.add((int(m[1]), m[2], 'RDR' if m[4] else 'regular'))
  return out or None

def parse_mech(note):
  """bags, heat and the wheel batch year / type / profile from one mech_notes block.  The wheels
  list wins over the wheel_type label (which is loose: 'halo' sits on Grapes and Devils too), and
  any field that is mixed across the set or unreadable comes back NA."""
  d = dict(re.findall(r'^(\w+):\s*(.*)$', note or '', re.M))
  out = {k: pd.to_numeric(d.get(k), errors='coerce') for k in ('bags', 'heat')}
  trips = _triples(d.get('wheels', '')) or _triples(d.get('wheel_type', ''))
  for i, k in enumerate(['wheel_year', 'wheel_type', 'wheel_profile']):
    vals = {t[i] for t in trips} if trips else set()
    out[k] = str(vals.pop()) if len(vals) == 1 else np.nan
  return out

mech = pd.DataFrame([parse_mech(n) for n in rolls['mech_notes']], index=rolls.index)
# the wheel columns stay object/NaN labels: plotly 6.3 cannot group a string-dtype column
# holding pd.NA, and the batch year is a label like raceday rather than a measurement
mech = mech.astype({'bags': 'Int64', 'heat': 'Int64'})

rolls = rolls.drop(columns=mech.columns, errors='ignore').join(mech)
cols = [c for c in rolls.columns if c not in mech.columns]
i = cols.index('mech_notes') + 1
rolls = rolls[cols[:i] + list(mech.columns) + cols[i:]]
rolls[['roll_id', 'date', 'buggy', 'mech_notes', *mech.columns]]

,roll_id,date,buggy,mech_notes,bags,heat,wheel_year,wheel_type,wheel_profile
0,1,2025-09-20,Zuke,,<NA>,<NA>,NaN,NaN,NaN
1,6,2025-09-21,Zuke,,<NA>,<NA>,NaN,NaN,NaN
2,7,2025-09-21,Zuke,,<NA>,<NA>,NaN,NaN,NaN
3,8,2025-09-21,Zuke,,<NA>,<NA>,NaN,NaN,NaN
4,9,2025-09-21,Inviscid,,<NA>,<NA>,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
1114,1412,2026-03-21,Inviscid,,<NA>,<NA>,NaN,NaN,NaN
1115,1413,2026-03-22,Kingpin II,no virb attached,<NA>,<NA>,NaN,NaN,NaN
1116,1414,2026-03-22,Seraph,no virb attached,<NA>,<NA>,NaN,NaN,NaN
1117,1415,2026-03-22,Seraph,no virb attached,<NA>,<NA>,NaN,NaN,NaN


In [70]:
from plotly.colors import sample_colorscale

RACEDAYS = sorted(rolls['raceday'].unique())
CMAPS = {  # defaults per color column: buggy keeps the map above, raceday runs old -> new
  'buggy': { 'Zuke': 'turquoise', 'Inviscid': 'black', 'Seraph': 'hotpink', 'Kingpin II': 'darkorange' },
  'raceday': dict(zip(RACEDAYS, sample_colorscale('Turbo', np.linspace(0.06, 0.94, len(RACEDAYS))))),
}

def plot_stats(x, y, hover=(), color=None, cmap=None, max_x_sd=None, df=None, x_sd=True, y_sd=True, **kwargs):
  """Scatter two `rolls` quantities against each other with +-2sd error bars."""
  d = (rolls if df is None else df).copy()
  if cmap is None and color in CMAPS:
    cmap = CMAPS[color]
    kwargs.setdefault('category_orders', {color: list(cmap)})
  if max_x_sd is not None:
    d = d[d[f'{x}_sd'] <= max_x_sd]
  err = {}
  if x_sd:
    axis, q = 'error_x', x
    if f'{q}_sd' in d:
      d[f'{q}_2sd'] = 2 * d[f'{q}_sd']
      err[axis] = f'{q}_2sd'
  if y_sd:
    axis, q = 'error_y', y
    if f'{q}_sd' in d:
      d[f'{q}_2sd'] = 2 * d[f'{q}_sd']
      err[axis] = f'{q}_2sd'
      
  fig = px.scatter(d, x=x, y=y, color=color, color_discrete_map=cmap or {},
                   hover_data=['roll_id', *hover], **err, **kwargs)
  return fig.update_traces(**{k: dict(thickness=1, width=0) for k in err})

In [71]:
rolls.columns

Index(['roll_id', 'date', 'raceday', 'roll_type', 'roll_number',
       'roll_start_time', 'driver', 'buggy', 'source', 'mech_notes', 'bags',
       'heat', 'wheel_year', 'wheel_type', 'wheel_profile', 'time', 'time_sd',
       'to_chute_time', 'to_chute_time_sd', 'chute_time', 'chute_time_sd',
       'to_stop_sign_time', 'to_stop_sign_time_sd', 'from_stop_sign_time',
       'from_stop_sign_time_sd', 'roll_time', 'roll_time_sd', 'hill_1_time',
       'hill_1_time_sd', 'hill_2_time', 'hill_2_time_sd', 'hill_3_time',
       'hill_3_time_sd', 'hill_4_time', 'hill_4_time_sd', 'hill_5_time',
       'hill_5_time_sd', 'max_speed', 'max_speed_sd', 'max_energy',
       'max_energy_sd', 'crosswalk_speed', 'crosswalk_speed_sd', 'chute_speed',
       'chute_speed_sd', 'crosswalk_energy', 'crosswalk_energy_sd',
       'chute_energy', 'chute_energy_sd', 'to_chute_energy_loss',
       'to_chute_energy_loss_sd', 'chute_energy_loss', 'chute_energy_loss_sd',
       'freeroll_energy_loss', 'freeroll_ener

In [73]:
plot_stats('crosswalk_energy', 'time', color='raceday', max_x_sd=2, x_sd=False)

In [75]:
plot_stats('crosswalk_speed', 'time', color='raceday', max_x_sd=0.4, x_sd=False)

In [92]:
plot_stats('crosswalk_speed', 'to_chute_time', color='raceday', max_x_sd=0.4, x_sd=False)

In [93]:
plot_stats('chute_speed', 'chute_energy_loss', color='raceday', max_x_sd=0.4, x_sd=False, y_sd=False)

In [39]:
plot_stats('crosswalk_energy', 'to_chute_energy_loss', color='raceday', max_x_sd=4, x_sd=False, y_sd=False)

In [49]:
plot_stats('crosswalk_speed', 'to_chute_energy_loss', color='raceday', max_x_sd=0.4, x_sd=False, y_sd=False)

In [ ]:
plot_stats('crosswalk_energy', 'to_chute_time', color='raceday', max_x_sd=4, x_sd=False, y_sd=False)

In [43]:
plot_stats('to_chute_time', 'chute_time', color='raceday', max_x_sd=4, x_sd=False, y_sd=False)

In [86]:
plot_stats('chute_speed', 'chute_time', color='raceday', max_x_sd=0.4, x_sd=False, y_sd=False)

In [64]:
plot_stats('to_stop_sign_time', 'from_stop_sign_time', color='raceday', max_x_sd=4, x_sd=False, y_sd=False)